# HPTS forecasting analysis

## Analysis

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / '.tools'))


from hashlib import sha256
from pathlib import Path
import json
import os
import sys
import warnings

_PACKAGE_ROOT = Path.cwd()
os.environ.setdefault("MPLCONFIGDIR", str(_PACKAGE_ROOT / ".matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
DATA = ROOT / "model_data.csv"
OUT = ROOT / "final_results"
FIG = OUT / "figures"
SEED = 77
REFIT_DAYS = 60
MIN_TRAIN_DAYS = 300

HAR = ["c_logrv_6", "c_logrv_24", "c_logrv_72", "c_logrv_168"]
IV_BTC = ["log_dvol_open", "dvol_chg_1_open", "dvol_chg_5_open", "iv_rv_gap_open"]
IV_ETH = ["log_dvol_eth_open", "dvol_eth_chg_1_open"]
IV_SPREAD = ["iv_spread_open"]
IV = IV_BTC + IV_ETH
XSEC = [
    "c_xs_logrv24", "c_xs_logrv24_rank", "c_xs_dispersion", "c_xs_breadth",
    "c_beta_mkt_168", "c_corr_mkt_168", "c_resid_ret_24", "c_mkt_ret_24",
]
TAIL = [
    "c_signed_jump_24", "c_jump_asym_24", "c_jump_share_24",
    "c_vol_ratio_6_72", "c_vol_ratio_24_168", "c_path_eff_24",
    "c_range_rel", "c_amihud_24",
]
SPOT = HAR + XSEC + TAIL
FULL = SPOT + IV

def load_and_audit() -> pd.DataFrame:
    df = pd.read_csv(DATA, parse_dates=["timestamp"])
    required = {"timestamp", "symbol", "split", "y", *FULL, *IV_SPREAD}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if df[list(required)].isna().any().any():
        raise ValueError("Model-ready columns contain missing values")
    if df.duplicated(["timestamp", "symbol"]).any():
        raise ValueError("Duplicate timestamp-symbol rows found")
    forbidden = [c for c in df if "dvol" in c.lower() and any(x in c.lower() for x in ("close", "high", "low"))]
    if forbidden:
        raise ValueError(f"Non-causal same-day DVOL fields found: {forbidden}")
    expected_split = np.select(
        [df.timestamp < "2023-01-01", df.timestamp < "2024-01-01"],
        ["train", "validation"], default="holdout",
    )
    if not np.array_equal(df.split.astype(str).to_numpy(), expected_split):
        raise ValueError("Stored split labels do not match the locked date protocol")
    return df.sort_values(["timestamp", "symbol"]).reset_index(drop=True)

def nw_t(x: pd.Series | np.ndarray, lags: int = 4) -> float:
    z = np.asarray(x, float)
    z = z[np.isfinite(z)]
    n = len(z)
    e = z - z.mean()
    long_var = e @ e / n
    for k in range(1, min(lags, n - 1) + 1):
        long_var += 2 * (1 - k / (lags + 1)) * (e[k:] @ e[:-k] / n)
    return float(z.mean() / np.sqrt(max(long_var, 1e-18) / n))

def two_sided_p(t: float) -> float:
    return float(2 * norm.sf(abs(t)))

def adjust_p(p: pd.Series | np.ndarray, method: str) -> np.ndarray:
    p = np.asarray(p, float)
    n = len(p)
    order = np.argsort(p)
    out = np.empty(n)
    if method == "holm":
        vals = np.maximum.accumulate(p[order] * (n - np.arange(n)))
    elif method == "bh":
        vals = p[order] * n / np.arange(1, n + 1)
        vals = np.minimum.accumulate(vals[::-1])[::-1]
    else:
        raise ValueError(method)
    out[order] = np.clip(vals, 0, 1)
    return out

def block_bootstrap_gain(frame: pd.DataFrame, model: str, comparator: str,
                         reps: int = 3000, block: int = 28) -> tuple[float, float]:
    daily = frame.assign(
        comparator_loss=(frame.y - frame[comparator]) ** 2,
        model_loss=(frame.y - frame[model]) ** 2,
    ).groupby("timestamp")[["comparator_loss", "model_loss"]].sum().to_numpy()
    n = len(daily)
    starts = np.arange(max(1, n - block + 1))
    rng = np.random.default_rng(SEED)
    draws = np.empty(reps)
    for r in range(reps):
        idx = np.concatenate([
            np.arange(s, min(s + block, n))
            for s in rng.choice(starts, int(np.ceil(n / block)), replace=True)
        ])[:n]
        loss = daily[idx].sum(0)
        draws[r] = 100 * (1 - loss[1] / loss[0])
    lo, hi = np.quantile(draws, [0.025, 0.975])
    return float(lo), float(hi)

def standardise(train: pd.DataFrame, test: pd.DataFrame, features: list[str]):
    x = train[features].to_numpy(float)
    z = test[features].to_numpy(float)
    mean = x.mean(0)
    std = x.std(0)
    std = np.where(std < 1e-12, 1.0, std)
    return (x - mean) / std, (z - mean) / std

def panel_design(train: pd.DataFrame, test: pd.DataFrame, features: list[str],
                 deviations: list[str], alpha: float, pool: float):
    x0, z0 = standardise(train, test, features)
    symbols = sorted(train.symbol.unique())
    symbol_map = {s: i for i, s in enumerate(symbols)}
    train_id = train.symbol.map(symbol_map).to_numpy()
    test_id = test.symbol.map(symbol_map).fillna(-1).astype(int).to_numpy()
    onehot_train = np.zeros((len(train), len(symbols)))
    onehot_test = np.zeros((len(test), len(symbols)))
    onehot_train[np.arange(len(train)), train_id] = 1
    valid = test_id >= 0
    onehot_test[np.arange(len(test))[valid], test_id[valid]] = 1

    x_parts = [np.ones((len(train), 1)), x0, onehot_train]
    z_parts = [np.ones((len(test), 1)), z0, onehot_test]
    penalties = [0.0] + [alpha] * len(features) + [0.25 * alpha] * len(symbols)
    if deviations:
        indices = [features.index(c) for c in deviations]
        dx = np.concatenate([onehot_train * x0[:, [j]] for j in indices], axis=1)
        dz = np.concatenate([onehot_test * z0[:, [j]] for j in indices], axis=1)
        x_parts.append(dx)
        z_parts.append(dz)
        penalties += [alpha * pool] * dx.shape[1]
    return np.concatenate(x_parts, 1), np.concatenate(z_parts, 1), np.asarray(penalties)

def panel_predict(train: pd.DataFrame, test: pd.DataFrame, features: list[str],
                  deviations: list[str], hp: dict) -> np.ndarray:
    x, z, penalty = panel_design(train, test, features, deviations, hp["alpha"], hp.get("pool", 1.0))
    beta = np.linalg.solve(x.T @ x + np.diag(penalty + 1e-10), x.T @ train.y.to_numpy(float))
    return z @ beta

def tune_panel(df: pd.DataFrame, features: list[str], deviations: list[str], model: str):
    train = df[df.timestamp < "2023-01-01"].reset_index(drop=True)
    validation = df[(df.timestamp >= "2023-01-01") & (df.timestamp < "2024-01-01")].reset_index(drop=True)
    pools = (1.0, 3.0, 10.0, 30.0) if deviations else (1.0,)
    rows = []
    for alpha in (3.0, 10.0, 30.0, 100.0):
        for pool in pools:
            hp = {"alpha": alpha, "pool": pool}
            pred = panel_predict(train, validation, features, deviations, hp)
            rows.append({"model": model, **hp, "validation_mse": np.mean((validation.y - pred) ** 2)})
    grid = pd.DataFrame(rows).sort_values("validation_mse").reset_index(drop=True)
    return grid, grid.iloc[0][["alpha", "pool", "validation_mse"]].to_dict()

def rolling_panel(df: pd.DataFrame, features: list[str], deviations: list[str],
                  hp: dict, name: str, window_days: int | None = None) -> pd.DataFrame:
    rows = []
    dates = np.array(sorted(df[df.timestamp >= "2023-01-01"].timestamp.unique()))
    for start in range(0, len(dates), REFIT_DAYS):
        block = dates[start:start + REFIT_DAYS]
        cutoff = block[0]
        train = df[df.timestamp < cutoff]
        if window_days is not None:
            train = train[train.timestamp >= cutoff - pd.Timedelta(days=window_days)]
        test = df[df.timestamp.isin(block)]
        pred = panel_predict(train.reset_index(drop=True), test.reset_index(drop=True), features, deviations, hp)
        rows.extend(zip(test.timestamp, test.symbol, test.y, pred))
    return pd.DataFrame(rows, columns=["timestamp", "symbol", "y", name])

def tune_per_asset_ridge(df: pd.DataFrame, features: list[str], model: str):
    train = df[df.timestamp < "2023-01-01"]
    validation = df[(df.timestamp >= "2023-01-01") & (df.timestamp < "2024-01-01")]
    rows = []
    for alpha in (0.1, 1.0, 10.0, 100.0):
        losses = []
        for symbol, group in train.groupby("symbol"):
            test = validation[validation.symbol == symbol]
            x, z = standardise(group, test, features)
            y = group.y.to_numpy(float)
            beta = np.linalg.solve(x.T @ x + alpha * np.eye(x.shape[1]), x.T @ (y - y.mean()))
            losses.extend((test.y.to_numpy() - (y.mean() + z @ beta)) ** 2)
        rows.append({"model": model, "alpha": alpha, "validation_mse": np.mean(losses)})
    grid = pd.DataFrame(rows).sort_values("validation_mse").reset_index(drop=True)
    return grid, float(grid.iloc[0].alpha)

def rolling_per_asset(df: pd.DataFrame, features: list[str], name: str,
                      kind: str = "ridge", params: dict | None = None) -> pd.DataFrame:
    params = params or {}
    rows = []
    dates = np.array(sorted(df.timestamp.unique()))
    for start in range(MIN_TRAIN_DAYS, len(dates), REFIT_DAYS):
        cutoff = dates[start]
        block = dates[start:start + REFIT_DAYS]
        for _, group in df.groupby("symbol"):
            train = group[group.timestamp < cutoff]
            test = group[group.timestamp.isin(block)]
            if len(train) < MIN_TRAIN_DAYS or test.empty:
                continue
            x, z = standardise(train, test, features)
            y = train.y.to_numpy(float)
            if kind == "elasticnet":
                fit = ElasticNet(alpha=params["alpha"], l1_ratio=params["l1_ratio"],
                                 max_iter=10000, random_state=SEED).fit(x, y)
                pred = fit.predict(z)
            elif kind == "hgb":
                fit = HistGradientBoostingRegressor(
                    max_iter=params["max_iter"], learning_rate=params["learning_rate"],
                    max_leaf_nodes=params["max_leaf_nodes"],
                    l2_regularization=params["l2_regularization"], random_state=SEED,
                ).fit(x, y)
                pred = fit.predict(z)
            else:
                alpha = params["alpha"]
                beta = np.linalg.solve(x.T @ x + alpha * np.eye(x.shape[1]), x.T @ (y - y.mean()))
                pred = y.mean() + z @ beta
            rows.extend(zip(test.timestamp, test.symbol, test.y, pred))
    return pd.DataFrame(rows, columns=["timestamp", "symbol", "y", name])

def build_predictions(df: pd.DataFrame):
    specs = {
        "Full_FixedEffects": (FULL, []),
        "HPTS_CommonIV": (FULL, SPOT),
        "HPTS_IVSpecific": (FULL, IV),
        "HPTS_Final": (FULL, FULL),
        "HPTS_NoIV": (SPOT, SPOT),
    }
    grids, choices, predictions = [], [], []
    for name, (features, deviations) in specs.items():
        print(f"Tuning and forecasting {name}", flush=True)
        grid, hp = tune_panel(df, features, deviations, name)
        grids.append(grid)
        choices.append({"model": name, **hp})
        predictions.append(rolling_panel(df, features, deviations, hp, name))

    per_asset_specs = {
        "HAR": HAR,
        "HAR_J": HAR + ["c_signed_jump_24", "c_jump_asym_24", "c_jump_share_24"],
        "HAR_IV_PerAsset": HAR + IV,
        "Full_PerAsset": FULL,
    }
    for name, features in per_asset_specs.items():
        grid, alpha = tune_per_asset_ridge(df, features, name)
        grids.append(grid)
        choices.append({"model": name, "alpha": alpha, "pool": np.nan,
                        "validation_mse": float(grid.iloc[0].validation_mse)})
        predictions.append(rolling_per_asset(df, features, name, params={"alpha": alpha}))

    predictions.append(rolling_per_asset(
        df, FULL, "ElasticNet", kind="elasticnet", params={"alpha": 0.1, "l1_ratio": 0.1}
    ))
    choices.append({"model": "ElasticNet", "alpha": 0.1, "pool": np.nan,
                    "validation_mse": np.nan, "l1_ratio": 0.1})
    predictions.append(rolling_per_asset(
        df, FULL, "HistGradientBoosting", kind="hgb",
        params={"max_iter": 200, "learning_rate": 0.03, "max_leaf_nodes": 7, "l2_regularization": 1.0},
    ))
    choices.append({"model": "HistGradientBoosting", "alpha": np.nan, "pool": np.nan,
                    "validation_mse": np.nan, "max_iter": 200, "learning_rate": 0.03,
                    "max_leaf_nodes": 7, "l2_regularization": 1.0})

    pred = predictions[0]
    for p in predictions[1:]:
        pred = pred.merge(p.drop(columns="y"), on=["timestamp", "symbol"], how="inner")
    pred = pred[pred.timestamp >= "2024-01-01"].reset_index(drop=True)
    lookup = df.set_index(["timestamp", "symbol"])["c_logrv_24"]
    pred["RandomWalk"] = [lookup.loc[(t, s)] for t, s in zip(pred.timestamp, pred.symbol)]
    return pred, pd.concat(grids, ignore_index=True), pd.DataFrame(choices), specs

def evaluate(pred: pd.DataFrame, subset: str, mask: np.ndarray | pd.Series | None = None):
    p = pred if mask is None else pred.loc[mask]
    models = [c for c in p if c not in ("timestamp", "symbol", "y")]
    base_mse = (p.y - p.HAR) ** 2
    base_qlike = np.exp(np.clip(p.y - p.HAR, -50, 50)) - (p.y - p.HAR) - 1
    rows = []
    for model in models:
        mse = (p.y - p[model]) ** 2
        qlike = np.exp(np.clip(p.y - p[model], -50, 50)) - (p.y - p[model]) - 1
        daily_mse = (base_mse - mse).groupby(p.timestamp).mean()
        daily_qlike = (base_qlike - qlike).groupby(p.timestamp).mean()
        tm, tq = nw_t(daily_mse), nw_t(daily_qlike)
        rows.append({
            "subset": subset, "model": model, "n": len(p),
            "mse": mse.mean(), "mse_gain_vs_har_pct": 100 * (1 - mse.sum() / base_mse.sum()),
            "mse_nw_t": tm, "mse_p_two_sided": two_sided_p(tm),
            "qlike": qlike.mean(), "qlike_gain_vs_har_pct": 100 * (1 - qlike.mean() / base_qlike.mean()),
            "qlike_nw_t": tq, "qlike_p_two_sided": two_sided_p(tq),
        })
    return pd.DataFrame(rows)

def direct_tests(pred: pd.DataFrame, subset: str, comparators: list[str]):
    p = pred if subset == "all_17_assets" else pred[~pred.symbol.isin(["BTCUSDT", "ETHUSDT"])]
    rows = []
    for comparator in comparators:
        a = (p.y - p[comparator]) ** 2
        b = (p.y - p.HPTS_Final) ** 2
        daily = (a - b).groupby(p.timestamp).mean()
        t = nw_t(daily)
        lo, hi = block_bootstrap_gain(p, "HPTS_Final", comparator)
        rows.append({
            "subset": subset, "model": "HPTS_Final", "comparator": comparator, "n": len(p),
            "gain_pct": 100 * (1 - b.sum() / a.sum()), "nw_t": t,
            "p_two_sided": two_sided_p(t), "bootstrap_95_low": lo, "bootstrap_95_high": hi,
        })
    out = pd.DataFrame(rows)
    out["p_holm"] = adjust_p(out.p_two_sided, "holm")
    return out

def per_asset_tests(pred: pd.DataFrame):
    rows = []
    for symbol, g in pred.groupby("symbol"):
        a = (g.y - g.HAR) ** 2
        b = (g.y - g.HPTS_Final) ** 2
        daily = (a - b).groupby(g.timestamp).mean()
        t = nw_t(daily)
        rows.append({"symbol": symbol, "n": len(g), "gain_pct": 100 * (1 - b.sum() / a.sum()),
                     "nw_t": t, "p_two_sided": two_sided_p(t)})
    out = pd.DataFrame(rows).sort_values("gain_pct", ascending=False).reset_index(drop=True)
    out["p_bh"] = adjust_p(out.p_two_sided, "bh")
    return out

def per_year(pred: pd.DataFrame):
    rows = []
    for year, g in pred.groupby(pred.timestamp.dt.year):
        base = ((g.y - g.HAR) ** 2).sum()
        for model in ("HPTS_Final", "HPTS_NoIV", "Full_PerAsset", "HistGradientBoosting"):
            rows.append({"year": year, "model": model,
                         "mse_gain_vs_har_pct": 100 * (1 - ((g.y - g[model]) ** 2).sum() / base)})
    return pd.DataFrame(rows)

def scale_diagnostic(pred: pd.DataFrame):
    y, har, final = pred.y.to_numpy(), pred.HAR.to_numpy(), pred.HPTS_Final.to_numpy()
    rows = []
    for scale, transform in (
        ("log_realised_variance", lambda x: x),
        ("log_realised_volatility", lambda x: x / 2),
        ("raw_realised_volatility", lambda x: np.exp(x / 2)),
    ):
        yt, ht, ft = transform(y), transform(har), transform(final)
        a, b = (yt - ht) ** 2, (yt - ft) ** 2
        daily = pd.Series(a - b).groupby(pred.timestamp).mean()
        t = nw_t(daily)
        rows.append({"scale": scale, "n": len(pred), "gain_vs_har_pct": 100 * (1 - b.sum() / a.sum()),
                     "nw_t": t, "p_two_sided": two_sided_p(t)})
    return pd.DataFrame(rows)

def robustness(df: pd.DataFrame, choices: pd.DataFrame, pred: pd.DataFrame):
    rows = []
    hp = choices.set_index("model").loc["HPTS_Final"].to_dict()
    work = df.copy()
    delayed = []
    for c in IV:
        name = f"lag1_{c}"
        work[name] = work.groupby("symbol")[c].shift(1)
        delayed.append(name)
    work = work.dropna(subset=delayed).reset_index(drop=True)

    variants = {
        "final_no_spread": (df, FULL, FULL, None),
        "with_iv_spread": (df, FULL + IV_SPREAD, FULL + IV_SPREAD, None),
        "btc_iv_only": (df, SPOT + IV_BTC, SPOT + IV_BTC, None),
        "eth_iv_only": (df, SPOT + IV_ETH, SPOT + IV_ETH, None),
        "iv_delayed_one_day": (work, SPOT + delayed, SPOT + delayed, None),
        "rolling_730_days": (df, FULL, FULL, 730),
        "rolling_1095_days": (df, FULL, FULL, 1095),
    }
    for label, (data, features, deviations, window) in variants.items():
        if label == "final_no_spread":
            q = pred[["timestamp", "symbol", "y", "HAR", "HPTS_Final"]].rename(columns={"HPTS_Final": label})
            chosen = hp
        else:
            grid, chosen = tune_panel(data, features, deviations, label)
            q = rolling_panel(data, features, deviations, chosen, label, window_days=window)
            q = q.merge(pred[["timestamp", "symbol", "HAR"]], on=["timestamp", "symbol"], how="inner")
            q = q[q.timestamp >= "2024-01-01"]
        a, b = (q.y - q.HAR) ** 2, (q.y - q[label]) ** 2
        daily = (a - b).groupby(q.timestamp).mean()
        t = nw_t(daily)
        rows.append({"variant": label, "n": len(q), "alpha": chosen["alpha"], "pool": chosen.get("pool", np.nan),
                     "gain_vs_har_pct": 100 * (1 - b.sum() / a.sum()), "nw_t": t,
                     "p_two_sided": two_sided_p(t)})
    return pd.DataFrame(rows).sort_values("gain_vs_har_pct", ascending=False)

def leave_one_asset_out_refit(df: pd.DataFrame, pred: pd.DataFrame, hp: dict):
    rows = []
    for omitted in sorted(df.symbol.unique()):
        print(f"Leave-one-asset-out refit: {omitted}", flush=True)
        sub = df[df.symbol != omitted].reset_index(drop=True)
        p = rolling_panel(sub, FULL, FULL, hp, "HPTS_Final")
        har = pred[pred.symbol != omitted][["timestamp", "symbol", "HAR"]]
        q = p.merge(har, on=["timestamp", "symbol"], how="inner")
        q = q[q.timestamp >= "2024-01-01"]
        a, b = (q.y - q.HAR) ** 2, (q.y - q.HPTS_Final) ** 2
        daily = (a - b).groupby(q.timestamp).mean()
        rows.append({"omitted_asset": omitted, "n": len(q), "gain_pct": 100 * (1 - b.sum() / a.sum()),
                     "nw_t": nw_t(daily)})
    return pd.DataFrame(rows)

def save_figures(model_table: pd.DataFrame, asset_table: pd.DataFrame, year_table: pd.DataFrame):
    plt.style.use("seaborn-v0_8-whitegrid")
    main = model_table[model_table.subset == "all_17_assets"].sort_values("mse_gain_vs_har_pct")
    colors = ["#0B6E4F" if m == "HPTS_Final" else "#8093A7" for m in main.model]
    fig, ax = plt.subplots(figsize=(8.2, 5.4))
    ax.barh(main.model, main.mse_gain_vs_har_pct, color=colors)
    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.set_xlabel("MSE improvement relative to HAR (%)")
    ax.set_title("HPTS-Final delivers the largest log-variance MSE improvement", loc="left", weight="bold")
    fig.tight_layout()
    fig.savefig(FIG / "figure_1_model_comparison.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    a = asset_table.sort_values("gain_pct")
    fig, ax = plt.subplots(figsize=(8.2, 5.2))
    ax.barh(a.symbol.str.replace("USDT", "", regex=False), a.gain_pct,
            color=np.where(a.gain_pct >= 0, "#0B6E4F", "#B44B4B"))
    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.set_xlabel("HPTS-Final MSE improvement relative to HAR (%)")
    ax.set_title("Cross-sectional consistency of the forecasting gain", loc="left", weight="bold")
    fig.tight_layout()
    fig.savefig(FIG / "figure_2_asset_gains.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.8, 4.6))
    for model, g in year_table.groupby("model"):
        ax.plot(g.year, g.mse_gain_vs_har_pct, marker="o", linewidth=2 if model == "HPTS_Final" else 1.2,
                label=model, color="#0B6E4F" if model == "HPTS_Final" else None)
    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set_xticks(sorted(year_table.year.unique()))
    ax.set_ylabel("MSE improvement relative to HAR (%)")
    ax.set_title("Forecast gains remain positive across holdout years", loc="left", weight="bold")
    ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(FIG / "figure_3_year_stability.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

def write_results_report(model_table: pd.DataFrame, direct: pd.DataFrame, no_options: pd.DataFrame,
                         asset: pd.DataFrame, years: pd.DataFrame, robust: pd.DataFrame,
                         scale: pd.DataFrame, choices: pd.DataFrame):
    overall = model_table[(model_table.subset == "all_17_assets") & (model_table.model == "HPTS_Final")].iloc[0]
    har = direct[(direct.subset == "all_17_assets") & (direct.comparator == "HAR")].iloc[0]
    no_har = no_options[no_options.comparator == "HAR"].iloc[0]
    no_iv = direct[(direct.subset == "all_17_assets") & (direct.comparator == "HPTS_NoIV")].iloc[0]
    positive = int((asset.gain_pct > 0).sum())
    significant = int((asset.p_bh < 0.05).sum())
    hp = choices.set_index("model").loc["HPTS_Final"]
    raw = scale[scale.scale == "raw_realised_volatility"].iloc[0]
    report = f"""# Final HPTS forecasting results

## Evaluation setup

- Target: log realised variance over the exact next 24 hours.
- Training: before 2023; validation: calendar 2023; untouched holdout: 2024 onward.
- Proposed model: one non-ensemble hierarchical ridge model.
- Final predictors: HAR, cross-sectional state, tail/path/liquidity, and causal BTC/ETH DVOL-open variables.
- BTC--ETH IV spread: omitted because the no-spread model had lower validation MSE.
- Selected HPTS penalty: alpha={hp.alpha:.0f}, pooling multiplier={hp.pool:.0f}.
- Random seed: {SEED}, used only by stochastic competitors and bootstrap inference; ridge estimates are deterministic.

## Results

HPTS-Final improves pooled holdout MSE relative to per-asset HAR by **{overall.mse_gain_vs_har_pct:.2f}%**
(Newey--West t={overall.mse_nw_t:.2f}, two-sided p={overall.mse_p_two_sided:.3g}).  The 28-day
block-bootstrap 95% interval is **[{har.bootstrap_95_low:.2f}%, {har.bootstrap_95_high:.2f}%]**.

On the 15 assets without listed options, the gain is **{no_har.gain_pct:.2f}%**
(t={no_har.nw_t:.2f}, p={no_har.p_two_sided:.3g}).  Relative to HPTS-NoIV, the full model gains
**{no_iv.gain_pct:.2f}%** (t={no_iv.nw_t:.2f}, p={no_iv.p_two_sided:.3g}).

The improvement is positive for **{positive}/17** assets and significant after Benjamini--Hochberg
correction for **{significant}/17** assets.  Leave-one-asset-out refits and timing/window robustness
are supplied as supplementary tables.

## Robustness

QLIKE is reported as a secondary diagnostic and is not the strongest result for HPTS.  Forecasting
log realised volatility is algebraically the same target up to a factor of one half and therefore
preserves the relative MSE gain.  On raw realised volatility, the HPTS gain falls to
**{raw.gain_vs_har_pct:.2f}%** (t={raw.nw_t:.2f}, p={raw.p_two_sided:.3g}); raw-scale superiority is
therefore not claimed.

Paired VaR/ES loss improvements were not statistically decisive and are not reported as primary results.
"""
    (ROOT / "FINAL_RESULTS.md").write_text(report, encoding="utf-8")

def write_reproducibility_metadata(df: pd.DataFrame, choices: pd.DataFrame):
    checksum = sha256(DATA.read_bytes()).hexdigest()
    audit = pd.DataFrame([
        {"check": "target_definition", "value": "log(sum of squared hourly log returns over next exact 24h)", "status": "pass"},
        {"check": "row_count", "value": len(df), "status": "pass"},
        {"check": "asset_count", "value": df.symbol.nunique(), "status": "pass"},
        {"check": "date_min", "value": df.timestamp.min().date(), "status": "pass"},
        {"check": "date_max", "value": df.timestamp.max().date(), "status": "pass"},
        {"check": "missing_model_values", "value": int(df[["y", *FULL]].isna().sum().sum()), "status": "pass"},
        {"check": "duplicate_asset_dates", "value": int(df.duplicated(["timestamp", "symbol"]).sum()), "status": "pass"},
        {"check": "forbidden_same_day_dvol_close_high_low", "value": 0, "status": "pass"},
        {"check": "model_data_sha256", "value": checksum, "status": "pass"},
    ])
    audit.to_csv(OUT / "data_and_leakage_audit.csv", index=False)
    clean_choices = []
    for row in choices.to_dict(orient="records"):
        clean_choices.append({key: (None if pd.isna(value) else value) for key, value in row.items()})
    metadata = {
        "seed": SEED,
        "target": "log realised variance over exact next 24 hours",
        "train_end_exclusive": "2023-01-01",
        "validation_start": "2023-01-01",
        "validation_end_exclusive": "2024-01-01",
        "holdout_start": "2024-01-01",
        "refit_days": REFIT_DAYS,
        "primary_loss": "squared error on log realised variance",
        "secondary_loss": "QLIKE on realised variance",
        "option_timing": "current UTC day DVOL open",
        "final_model": "HPTS_Final, hierarchical ridge, no IV spread",
        "model_data_sha256": checksum,
        "selected_hyperparameters": clean_choices,
    }
    (OUT / "run_metadata.json").write_text(
        json.dumps(metadata, indent=2, default=str, allow_nan=False), encoding="utf-8"
    )

def main():
    OUT.mkdir(parents=True, exist_ok=True)
    FIG.mkdir(parents=True, exist_ok=True)
    df = load_and_audit()
    print(f"Loaded {len(df):,} rows, {df.symbol.nunique()} assets", flush=True)

    pred, grids, choices, _ = build_predictions(df)
    model_table = pd.concat([
        evaluate(pred, "all_17_assets"),
        evaluate(pred, "15_assets_without_listed_options", ~pred.symbol.isin(["BTCUSDT", "ETHUSDT"])),
    ], ignore_index=True)

    comparators = [
        "HAR", "RandomWalk", "HAR_J", "HAR_IV_PerAsset", "HPTS_NoIV",
        "Full_FixedEffects", "Full_PerAsset", "HPTS_CommonIV", "HPTS_IVSpecific",
        "ElasticNet", "HistGradientBoosting",
    ]
    direct_all = direct_tests(pred, "all_17_assets", comparators)
    direct_no_options = direct_tests(
        pred, "15_assets_without_listed_options",
        ["HAR", "HPTS_NoIV", "Full_FixedEffects", "Full_PerAsset", "HPTS_CommonIV", "HPTS_IVSpecific"],
    )
    asset = per_asset_tests(pred)
    years = per_year(pred)
    scale = scale_diagnostic(pred)
    robust = robustness(df, choices, pred)
    hp = choices.set_index("model").loc["HPTS_Final"].to_dict()
    loao = leave_one_asset_out_refit(df, pred, hp)

    model_table.sort_values(["subset", "mse_gain_vs_har_pct"], ascending=[True, False]).to_csv(OUT / "table_1_model_comparison.csv", index=False)
    direct_all.to_csv(OUT / "table_2_direct_tests.csv", index=False)
    direct_no_options.to_csv(OUT / "table_3_no_options_assets.csv", index=False)
    asset.to_csv(OUT / "table_4_per_asset.csv", index=False)
    robust.to_csv(OUT / "table_5_robustness.csv", index=False)
    scale.to_csv(OUT / "table_6_target_scale_diagnostic.csv", index=False)
    years.to_csv(OUT / "supplement_year_stability.csv", index=False)
    loao.to_csv(OUT / "supplement_leave_one_asset_out_refit.csv", index=False)
    grids.to_csv(OUT / "validation_grids.csv", index=False)
    choices.to_csv(OUT / "selected_hyperparameters.csv", index=False)
    pred.to_csv(OUT / "holdout_predictions.csv", index=False)

    save_figures(model_table, asset, years)
    write_results_report(model_table, direct_all, direct_no_options, asset, years, robust, scale, choices)
    write_reproducibility_metadata(df, choices)
    print("Final analysis completed successfully.", flush=True)

def finalize_from_saved_results():

    OUT.mkdir(parents=True, exist_ok=True)
    FIG.mkdir(parents=True, exist_ok=True)
    df = load_and_audit()
    model_table = pd.read_csv(OUT / "table_1_model_comparison.csv")
    direct_all = pd.read_csv(OUT / "table_2_direct_tests.csv")
    direct_no_options = pd.read_csv(OUT / "table_3_no_options_assets.csv")
    asset = pd.read_csv(OUT / "table_4_per_asset.csv")
    robust = pd.read_csv(OUT / "table_5_robustness.csv")
    scale = pd.read_csv(OUT / "table_6_target_scale_diagnostic.csv")
    years = pd.read_csv(OUT / "supplement_year_stability.csv")
    choices = pd.read_csv(OUT / "selected_hyperparameters.csv")
    save_figures(model_table, asset, years)
    write_results_report(model_table, direct_all, direct_no_options, asset, years, robust, scale, choices)
    write_reproducibility_metadata(df, choices)
    print("Finalization from saved results completed successfully.", flush=True)


In [ ]:
main()

## Primary results

In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((ROOT / 'FINAL_RESULTS.md').read_text(encoding='utf-8')))
display(pd.read_csv(OUT / 'table_1_model_comparison.csv')
        .query("subset == 'all_17_assets'")
        .sort_values('mse_gain_vs_har_pct', ascending=False))
display(Image(filename=str(FIG / 'figure_1_model_comparison.png')))

## Robustness analyses

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / '.tools'))


from pathlib import Path
import json
import os
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor

import run_final_analysis as core

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
OUT = ROOT / "acceptance_extensions"
FIG = OUT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
RNG = np.random.default_rng(core.SEED)

def performance(frame: pd.DataFrame, model: str, comparator: str = "HAR") -> dict:
    a = (frame.y - frame[comparator]) ** 2
    b = (frame.y - frame[model]) ** 2
    daily = (a - b).groupby(frame.timestamp).mean()
    t = core.nw_t(daily)
    return {
        "model": model, "comparator": comparator, "n": len(frame),
        "mse_gain_pct": 100 * (1 - b.sum() / a.sum()),
        "nw_t": t, "p_two_sided": core.two_sided_p(t),
    }

def panel_hgb_design(train: pd.DataFrame, test: pd.DataFrame):
    x, z = core.standardise(train, test, core.FULL)
    symbols = sorted(train.symbol.unique())
    ids = {s: i for i, s in enumerate(symbols)}
    tr_id = train.symbol.map(ids).to_numpy()
    te_id = test.symbol.map(ids).fillna(-1).astype(int).to_numpy()
    tr_oh = np.zeros((len(train), len(symbols)))
    te_oh = np.zeros((len(test), len(symbols)))
    tr_oh[np.arange(len(train)), tr_id] = 1
    valid = te_id >= 0
    te_oh[np.arange(len(test))[valid], te_id[valid]] = 1
    return np.c_[x, tr_oh], np.c_[z, te_oh]

def fit_panel_hgb(train, test, hp):
    x, z = panel_hgb_design(train, test)
    fit = HistGradientBoostingRegressor(
        max_iter=hp["max_iter"], learning_rate=hp["learning_rate"],
        max_leaf_nodes=hp["max_leaf_nodes"], min_samples_leaf=hp["min_samples_leaf"],
        l2_regularization=hp["l2_regularization"], random_state=core.SEED,
    ).fit(x, train.y.to_numpy(float))
    return fit.predict(z)

def pooled_nonlinear(df: pd.DataFrame, base: pd.DataFrame):
    train = df[df.timestamp < "2023-01-01"].reset_index(drop=True)
    val = df[(df.timestamp >= "2023-01-01") & (df.timestamp < "2024-01-01")].reset_index(drop=True)
    configs = [
        {"max_iter": 150, "learning_rate": .03, "max_leaf_nodes": 7, "min_samples_leaf": 40, "l2_regularization": 3.0},
        {"max_iter": 250, "learning_rate": .03, "max_leaf_nodes": 15, "min_samples_leaf": 40, "l2_regularization": 3.0},
        {"max_iter": 200, "learning_rate": .05, "max_leaf_nodes": 7, "min_samples_leaf": 80, "l2_regularization": 10.0},
        {"max_iter": 250, "learning_rate": .03, "max_leaf_nodes": 31, "min_samples_leaf": 80, "l2_regularization": 10.0},
    ]
    grid = []
    for i, hp in enumerate(configs):
        pred = fit_panel_hgb(train, val, hp)
        grid.append({"config": i, **hp, "validation_mse": np.mean((val.y.to_numpy() - pred) ** 2)})
    grid = pd.DataFrame(grid).sort_values("validation_mse")
    grid.to_csv(OUT / "nonlinear_validation_grid.csv", index=False)
    hp = configs[int(grid.iloc[0].config)]
    rows = []
    dates = np.array(sorted(df[df.timestamp >= "2024-01-01"].timestamp.unique()))
    for start in range(0, len(dates), core.REFIT_DAYS):
        block = dates[start:start + core.REFIT_DAYS]
        cutoff = block[0]
        tr = df[df.timestamp < cutoff].reset_index(drop=True)
        te = df[df.timestamp.isin(block)].reset_index(drop=True)
        pr = fit_panel_hgb(tr, te, hp)
        rows.extend(zip(te.timestamp, te.symbol, te.y, pr))
    q = pd.DataFrame(rows, columns=["timestamp", "symbol", "y", "Pooled_HGB"])
    q = q.merge(base[["timestamp", "symbol", "HAR", "HPTS_Final", "Full_PerAsset"]],
                on=["timestamp", "symbol"], how="inner")
    q.to_csv(OUT / "pooled_nonlinear_predictions.csv", index=False)
    return pd.DataFrame([
        performance(q, "Pooled_HGB"),
        performance(q, "HPTS_Final", "Pooled_HGB"),
        performance(q, "Full_PerAsset", "Pooled_HGB"),
    ]), hp

def make_stale(df: pd.DataFrame, days: int):
    work = df.copy()
    names = []
    for c in core.IV:
        name = f"stale{days}_{c}"
        work[name] = work.groupby("symbol")[c].shift(days)
        names.append(name)
    return work.dropna(subset=names).reset_index(drop=True), names

def make_permuted(df: pd.DataFrame):
    work = df.copy()
    pure = ["log_dvol_open", "dvol_chg_1_open", "dvol_chg_5_open",
            "log_dvol_eth_open", "dvol_eth_chg_1_open"]
    names = [f"perm_{c}" for c in pure] + ["perm_iv_rv_gap_open"]
    for name in names:
        work[name] = np.nan

    for split, idx in work.groupby("split").groups.items():
        part = work.loc[idx, ["timestamp", *pure]].drop_duplicates("timestamp").sort_values("timestamp")
        source = np.arange(len(part))
        local = np.random.default_rng(core.SEED + len(split))
        local.shuffle(source)
        for c in pure:
            mapper = dict(zip(part.timestamp, part[c].to_numpy()[source]))
            work.loc[idx, f"perm_{c}"] = work.loc[idx, "timestamp"].map(mapper)
    work["perm_iv_rv_gap_open"] = work["perm_log_dvol_open"] - work["c_logrv_24"]
    return work.dropna(subset=names).reset_index(drop=True), names

def placebos(df: pd.DataFrame, base: pd.DataFrame, hp: dict):
    rows = [performance(base, "HPTS_Final") | {"variant": "current_open"}]
    pred_out = base[["timestamp", "symbol", "y", "HAR", "HPTS_Final"]].copy()
    variants = []
    for lag in (1, 2, 3, 5, 7):
        work, iv = make_stale(df, lag)
        variants.append((f"dvol_stale_{lag}d", work, core.SPOT + iv))
    work, iv = make_permuted(df)
    variants.append(("dvol_date_permuted", work, core.SPOT + iv))
    for label, work, features in variants:
        q = core.rolling_panel(work, features, features, hp, label)
        q = q[q.timestamp >= "2024-01-01"].merge(
            base[["timestamp", "symbol", "HAR"]], on=["timestamp", "symbol"], how="inner")
        rows.append(performance(q, label) | {"variant": label})
        pred_out = pred_out.merge(q[["timestamp", "symbol", label]], on=["timestamp", "symbol"], how="left")
    result = pd.DataFrame(rows)
    result.to_csv(OUT / "dvol_placebo_results.csv", index=False)
    pred_out.to_csv(OUT / "dvol_placebo_predictions.csv", index=False)
    direct_rows = []
    joined = base[["timestamp", "symbol", "y", "HPTS_Final", "HPTS_NoIV"]].merge(
        pred_out.drop(columns=["y", "HAR", "HPTS_Final"]), on=["timestamp", "symbol"], how="inner")
    for comparator in ["HPTS_NoIV", *[c for c in joined if c.startswith("dvol_")]]:
        a = (joined.y - joined[comparator]) ** 2
        b = (joined.y - joined.HPTS_Final) ** 2
        daily = (a - b).groupby(joined.timestamp).mean()
        t = core.nw_t(daily)
        direct_rows.append({"comparator": comparator, "gain_current_pct": 100 * (1 - b.sum() / a.sum()),
                            "nw_t": t, "p_two_sided": core.two_sided_p(t)})
    direct = pd.DataFrame(direct_rows)
    direct["holm_p"] = core.adjust_p(direct.p_two_sided, "holm")
    direct.to_csv(OUT / "dvol_direct_comparisons.csv", index=False)
    perm = "dvol_date_permuted"
    a = (joined.y - joined.HPTS_NoIV) ** 2
    b = (joined.y - joined[perm]) ** 2
    daily = (a - b).groupby(joined.timestamp).mean()
    t = core.nw_t(daily)
    pd.DataFrame([{"comparison": "date-permuted DVOL vs HPTS-NoIV",
                   "gain_pct": 100 * (1 - b.sum() / a.sum()), "nw_t": t,
                   "p_two_sided": core.two_sided_p(t)}]).to_csv(OUT / "dvol_permuted_vs_noiv.csv", index=False)
    return result

def block_analysis(df: pd.DataFrame, base: pd.DataFrame):
    blocks = {"HAR": core.HAR, "XSEC": core.XSEC, "TAIL": core.TAIL, "IV": core.IV}
    rows, grids = [], []
    for omitted, cols in blocks.items():
        features = [c for c in core.FULL if c not in cols]
        label = f"HPTS_without_{omitted}"
        grid, hp = core.tune_panel(df, features, features, label)
        grids.append(grid)
        q = core.rolling_panel(df, features, features, hp, label)
        q = q[q.timestamp >= "2024-01-01"].merge(
            base[["timestamp", "symbol", "HAR", "HPTS_Final"]], on=["timestamp", "symbol"], how="inner")
        row = performance(q, "HPTS_Final", label)
        row.update({"omitted_block": omitted, "ablation_model": label,
                    "ablated_gain_vs_har_pct": performance(q, label)["mse_gain_pct"],
                    "alpha": hp["alpha"], "pool": hp["pool"]})
        rows.append(row)
    pd.concat(grids).to_csv(OUT / "block_ablation_validation_grids.csv", index=False)
    out = pd.DataFrame(rows)
    out.to_csv(OUT / "block_ablation_results.csv", index=False)

    coef_rows = []
    dates = np.array(sorted(df[df.timestamp >= "2024-01-01"].timestamp.unique()))
    hp = {"alpha": 100.0, "pool": 10.0}
    for start in range(0, len(dates), core.REFIT_DAYS):
        cutoff = dates[start]
        tr = df[df.timestamp < cutoff].reset_index(drop=True)
        te = df[df.timestamp.isin(dates[start:start + core.REFIT_DAYS])].reset_index(drop=True)
        x, _, penalty = core.panel_design(tr, te, core.FULL, core.FULL, hp["alpha"], hp["pool"])
        beta = np.linalg.solve(x.T @ x + np.diag(penalty + 1e-10), x.T @ tr.y.to_numpy(float))
        p, a = len(core.FULL), tr.symbol.nunique()
        common = beta[1:1+p]
        dev = beta[1+p+a:].reshape(p, a)
        for block, cols in blocks.items():
            idx = [core.FULL.index(c) for c in cols]
            coef_rows.append({"cutoff": cutoff, "block": block,
                              "common_abs_mean": np.mean(np.abs(common[idx])),
                              "deviation_rms": np.sqrt(np.mean(dev[idx] ** 2)),
                              "total_rms": np.sqrt(np.mean(common[idx] ** 2) + np.mean(dev[idx] ** 2))})
    coef = pd.DataFrame(coef_rows)
    coef.to_csv(OUT / "coefficient_block_summary_by_refit.csv", index=False)
    summary = coef.groupby("block")[["common_abs_mean", "deviation_rms", "total_rms"]].agg(["mean", "std"])
    summary.to_csv(OUT / "coefficient_block_summary.csv")
    return out, coef

def economic_value(df: pd.DataFrame, base: pd.DataFrame):
    future = df.sort_values(["symbol", "timestamp"])[["timestamp", "symbol", "ret_24"]].copy()
    future["future_return"] = future.groupby("symbol").ret_24.shift(-1)
    q = base.merge(future[["timestamp", "symbol", "future_return"]], on=["timestamp", "symbol"], how="left").dropna()
    target = 0.02
    rows = []
    for cost_bps in (0, 5, 10, 20):
        strategies = {}
        for model in ("HAR", "HPTS_Final"):
            vol = np.sqrt(np.exp(np.clip(q[model].to_numpy(), -30, 10)))
            weight = np.clip(target / vol, 0, 2)
            tmp = q[["timestamp", "symbol"]].copy()
            tmp["weight"] = weight
            tmp["turnover"] = tmp.groupby("symbol").weight.diff().abs().fillna(0)
            net = weight * q.future_return.to_numpy() - cost_bps / 10000 * tmp.turnover.to_numpy()
            strategies[model] = (weight, net)
            daily = pd.DataFrame({"timestamp": q.timestamp, "net": net}).groupby("timestamp").mean().net
            mean, var = daily.mean(), daily.var(ddof=1)
            rows.append({"cost_bps": cost_bps, "model": model, "n": len(q),
                         "annual_return": 365 * mean, "annual_volatility": np.sqrt(365 * var),
                         "sharpe": np.sqrt(365) * mean / np.sqrt(var),
                         "certainty_equivalent_gamma3": 365 * (mean - 1.5 * var),
                         "mean_turnover": pd.Series(tmp.turnover).mean(),
                         "variance_target_mae": np.mean(np.abs((weight * q.future_return.to_numpy()) ** 2 - target ** 2))})
        h = strategies["HPTS_Final"][1]
        b = strategies["HAR"][1]
        util_diff = pd.Series((h - 1.5*h*h) - (b - 1.5*b*b)).groupby(q.timestamp).mean()
        t = core.nw_t(util_diff)
        rows[-1]["utility_diff_nw_t_vs_har"] = t
        rows[-1]["utility_diff_p_vs_har"] = core.two_sided_p(t)
    out = pd.DataFrame(rows)
    out.to_csv(OUT / "economic_value_volatility_timing.csv", index=False)
    return out

def figures(placebo, nonlinear, ablation, coef):
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=(8, 4.8))
    p = placebo.sort_values("mse_gain_pct")
    ax.barh(p.variant, p.mse_gain_pct, color=["#0B6E4F" if x == "current_open" else "#8093A7" for x in p.variant])
    ax.set_xlabel("MSE improvement relative to HAR (%)")
    ax.set_title("DVOL timing and permutation diagnostics", loc="left", weight="bold")
    fig.tight_layout(); fig.savefig(FIG / "figure_E1_dvol_placebos.png", dpi=220); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    a = ablation.sort_values("mse_gain_pct")
    ax.barh(a.omitted_block, a.mse_gain_pct, color="#2E74B5")
    ax.axvline(0, color="black", lw=.8)
    ax.set_xlabel("HPTS-Final MSE gain over the ablated refit (%)")
    ax.set_title("Incremental contribution of predictor blocks", loc="left", weight="bold")
    fig.tight_layout(); fig.savefig(FIG / "figure_E2_block_ablation.png", dpi=220); plt.close(fig)

    c = coef.groupby("block").total_rms.mean().sort_values()
    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    ax.barh(c.index, c.values, color="#0B6E4F")
    ax.set_xlabel("Mean standardised coefficient RMS")
    ax.set_title("Coefficient magnitude by predictor block", loc="left", weight="bold")
    fig.tight_layout(); fig.savefig(FIG / "figure_E3_coefficient_blocks.png", dpi=220); plt.close(fig)

def coefficient_only(df: pd.DataFrame):
    blocks = {"HAR": core.HAR, "XSEC": core.XSEC, "TAIL": core.TAIL, "IV": core.IV}
    coef_rows = []
    dates = np.array(sorted(df[df.timestamp >= "2024-01-01"].timestamp.unique()))
    hp = {"alpha": 100.0, "pool": 10.0}
    for start in range(0, len(dates), core.REFIT_DAYS):
        cutoff = dates[start]
        tr = df[df.timestamp < cutoff].reset_index(drop=True)
        te = df[df.timestamp.isin(dates[start:start + core.REFIT_DAYS])].reset_index(drop=True)
        x, _, penalty = core.panel_design(tr, te, core.FULL, core.FULL, hp["alpha"], hp["pool"])
        beta = np.linalg.solve(x.T @ x + np.diag(penalty + 1e-10), x.T @ tr.y.to_numpy(float))
        p, a = len(core.FULL), tr.symbol.nunique()
        common = beta[1:1+p]
        dev = beta[1+p+a:].reshape(p, a)
        for block, cols in blocks.items():
            idx = [core.FULL.index(c) for c in cols]
            coef_rows.append({"cutoff": cutoff, "block": block,
                              "common_abs_mean": np.mean(np.abs(common[idx])),
                              "deviation_rms": np.sqrt(np.mean(dev[idx] ** 2)),
                              "total_rms": np.sqrt(np.mean(common[idx] ** 2) + np.mean(dev[idx] ** 2))})
    coef = pd.DataFrame(coef_rows)
    coef.to_csv(OUT / "coefficient_block_summary_by_refit.csv", index=False)
    coef.groupby("block")[["common_abs_mean", "deviation_rms", "total_rms"]].agg(["mean", "std"]).to_csv(
        OUT / "coefficient_block_summary.csv")
    return coef

def main():
    df = core.load_and_audit()
    base = pd.read_csv(core.OUT / "holdout_predictions.csv", parse_dates=["timestamp"])
    choices = pd.read_csv(core.OUT / "selected_hyperparameters.csv").set_index("model")
    hp = {"alpha": float(choices.loc["HPTS_Final", "alpha"]), "pool": float(choices.loc["HPTS_Final", "pool"])}
    print("1/4 economic-value diagnostic", flush=True)
    econ = economic_value(df, base)
    print("2/4 pooled nonlinear benchmark", flush=True)
    nonlinear, nonlinear_hp = pooled_nonlinear(df, base)
    nonlinear.to_csv(OUT / "pooled_nonlinear_results.csv", index=False)
    print("3/4 DVOL placebos", flush=True)
    placebo = placebos(df, base, hp)
    print("4/4 block attribution", flush=True)
    ablation, coef = block_analysis(df, base)
    figures(placebo, nonlinear, ablation, coef)
    meta = {"seed": core.SEED, "holdout_start": "2024-01-01", "nonlinear_hp": nonlinear_hp,
            "note": "All nonlinear tuning used calendar 2023; stale/permuted tests reuse pre-selected HPTS hyperparameters."}
    (OUT / "run_metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print("\nNONLINEAR\n", nonlinear.to_string(index=False))
    print("\nPLACEBOS\n", placebo.to_string(index=False))
    print("\nABLATIONS\n", ablation.to_string(index=False))
    print("\nECONOMIC\n", econ.to_string(index=False))


In [ ]:
from IPython.display import display, Image
extension_dir = Path.cwd() / 'acceptance_extensions'
display(pd.read_csv(extension_dir / 'pooled_nonlinear_results.csv'))
display(pd.read_csv(extension_dir / 'dvol_direct_comparisons.csv'))
display(pd.read_csv(extension_dir / 'block_ablation_results_with_holm.csv'))
display(pd.read_csv(extension_dir / 'economic_value_volatility_timing.csv'))
display(Image(filename=str(extension_dir / 'figures' / 'figure_E2_block_ablation.png')))

## Additional benchmarks and horizon tests

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / '.tools'))


import argparse
import json
import os
import sys
import warnings
from pathlib import Path

SCRIPT_DIR = Path.cwd()
ROOT = SCRIPT_DIR.parent
PACKAGE = SCRIPT_DIR if (SCRIPT_DIR / "model_data.csv").exists() else ROOT / "HPTS_reproducible_package"
OUT = PACKAGE / "requested_tests" if PACKAGE == SCRIPT_DIR else ROOT / "experiments_seed77" / "results"
os.environ.setdefault("MPLCONFIGDIR", str(OUT / ".matplotlib"))
sys.path.insert(0, str(PACKAGE))

import numpy as np
import pandas as pd
from arch import arch_model
from scipy.stats import norm
from statsmodels.tsa.arima.model import ARIMA

import run_final_analysis as core

warnings.filterwarnings("ignore")

SEED = 77
REPS = 1000
RAW_PANEL = Path(os.environ.get("HPTS_RAW_PANEL", "panel_hourly.parquet"))
PURE_IV = [
    "log_dvol_open", "dvol_chg_1_open", "dvol_chg_5_open",
    "log_dvol_eth_open", "dvol_eth_chg_1_open",
]

def nw_t(values, lags=4):
    x = np.asarray(values, float)
    x = x[np.isfinite(x)]
    e = x - x.mean()
    n = len(x)
    v = e @ e / n
    for lag in range(1, min(lags, n - 1) + 1):
        v += 2 * (1 - lag / (lags + 1)) * (e[lag:] @ e[:-lag] / n)
    return float(x.mean() / np.sqrt(max(v, 1e-18) / n))

def score(frame, model, comparator="HAR", horizon_days=1):
    a = (frame.y - frame[comparator]) ** 2
    b = (frame.y - frame[model]) ** 2
    daily = (a - b).groupby(frame.timestamp).mean()
    lag = max(4, horizon_days - 1)
    t = nw_t(daily, lag)
    return {
        "model": model,
        "comparator": comparator,
        "n": len(frame),
        "mse": float(b.mean()),
        "gain_pct": float(100 * (1 - b.sum() / a.sum())),
        "nw_lags": lag,
        "nw_t": t,
        "p_two_sided": float(2 * norm.sf(abs(t))),
    }

def repeated_permutation():
    df = core.load_and_audit()
    base = pd.read_csv(PACKAGE / "final_results" / "holdout_predictions.csv",
                       parse_dates=["timestamp"])
    hold = df[df.timestamp >= "2024-01-01"].copy().reset_index(drop=True)
    hold = hold.merge(base[["timestamp", "symbol", "y", "HAR", "HPTS_Final", "HPTS_NoIV"]],
                      on=["timestamp", "symbol"], suffixes=("", "_saved"), how="inner")
    dates = np.array(sorted(hold.timestamp.unique()))
    date_id = pd.Categorical(hold.timestamp, categories=dates, ordered=True).codes
    symbols = sorted(df.symbol.unique())
    symbol_id = pd.Categorical(hold.symbol, categories=symbols, ordered=True).codes.astype(int)
    pure_cube = np.empty((len(dates), len(symbols), len(PURE_IV)))
    dated = hold.set_index(["timestamp", "symbol"])
    for j, col in enumerate(PURE_IV):
        btc = dated[col].unstack("symbol").reindex(dates)["BTCUSDT"].to_numpy(float)
        eth = dated[col].unstack("symbol").reindex(dates)["ETHUSDT"].to_numpy(float)
        pure_cube[:, :, j] = btc[:, None]
        if col in PURE_IV[:3]:
            pure_cube[:, symbols.index("ETHUSDT"), j] = eth

    correct = hold.HPTS_Final.to_numpy(float)
    y = hold.y_saved.to_numpy(float)
    noiv = hold.HPTS_NoIV.to_numpy(float)
    effects = np.zeros((len(hold), len(core.IV)))
    correct_std = np.zeros_like(effects)

    hp = {"alpha": 100.0, "pool": 10.0}
    block_dates = dates
    for start in range(0, len(block_dates), core.REFIT_DAYS):
        block = block_dates[start:start + core.REFIT_DAYS]
        cutoff = block[0]
        train = df[df.timestamp < cutoff].reset_index(drop=True)
        test_idx = np.flatnonzero(hold.timestamp.isin(block).to_numpy())
        test = hold.iloc[test_idx].copy().reset_index(drop=True)
        x, z, penalty = core.panel_design(train, test, core.FULL, core.FULL,
                                          hp["alpha"], hp["pool"])
        beta = np.linalg.solve(x.T @ x + np.diag(penalty + 1e-10),
                               x.T @ train.y.to_numpy(float))
        means = train[core.FULL].mean().to_numpy(float)
        stds = train[core.FULL].std(ddof=0).replace(0, 1).to_numpy(float)
        ns = len(symbols)
        dev0 = 1 + len(core.FULL) + ns
        for j, col in enumerate(core.IV):
            k = core.FULL.index(col)
            effective = beta[1 + k] + beta[dev0 + k * ns + symbol_id[test_idx]]
            effects[test_idx, j] = effective / stds[k]
            correct_std[test_idx, j] = test[col].to_numpy(float)

    rng = np.random.default_rng(SEED)
    observed_gain = 100 * (1 - np.sum((y - correct) ** 2) / np.sum((y - noiv) ** 2))
    perm_gain = np.empty(REPS)
    current_vs_perm = np.empty(REPS)
    correct_loss = np.sum((y - correct) ** 2)
    noiv_loss = np.sum((y - noiv) ** 2)
    for rep in range(REPS):
        order = rng.permutation(len(dates))
        vals = pure_cube[order[date_id], symbol_id, :]
        perm = np.column_stack([
            vals[:, 0], vals[:, 1], vals[:, 2],
            vals[:, 0] - hold.c_logrv_24.to_numpy(float), vals[:, 3], vals[:, 4],
        ])
        pred = correct + np.sum(effects * (perm - correct_std), axis=1)
        loss = np.sum((y - pred) ** 2)
        perm_gain[rep] = 100 * (1 - loss / noiv_loss)
        current_vs_perm[rep] = 100 * (1 - correct_loss / loss)

    summary = {
        "seed": SEED,
        "permutations": REPS,
        "test": "locked-model joint-date permutation of the public IV state on the untouched holdout",
        "observed_HPTS_Final_gain_vs_NoIV_pct": observed_gain,
        "permuted_gain_vs_NoIV_mean_pct": float(perm_gain.mean()),
        "permuted_gain_vs_NoIV_q025_pct": float(np.quantile(perm_gain, .025)),
        "permuted_gain_vs_NoIV_q975_pct": float(np.quantile(perm_gain, .975)),
        "permutation_p_one_sided": float((1 + np.sum(perm_gain >= observed_gain)) / (REPS + 1)),
        "current_gain_vs_permuted_mean_pct": float(current_vs_perm.mean()),
        "current_gain_vs_permuted_q025_pct": float(np.quantile(current_vs_perm, .025)),
        "current_gain_vs_permuted_q975_pct": float(np.quantile(current_vs_perm, .975)),
    }
    pd.DataFrame({"rep": np.arange(REPS), "permuted_gain_vs_NoIV_pct": perm_gain,
                  "current_gain_vs_permuted_pct": current_vs_perm}).to_csv(
        OUT / "permutation_draws.csv", index=False)
    (OUT / "permutation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2), flush=True)

def fractional_weights(d, length, threshold=1e-5):
    w = [1.0]
    for k in range(1, length):
        w.append(-w[-1] * (d - k + 1) / k)
        if k > 50 and abs(w[-1]) < threshold:
            break
    return np.asarray(w)

def gph_d(x):
    x = np.asarray(x, float)
    n = len(x)
    m = max(12, int(n ** .55))
    ft = np.fft.fft(x - x.mean())
    periodogram = np.abs(ft) ** 2 / (2 * np.pi * n)
    lam = 2 * np.pi * np.arange(1, m + 1) / n
    slope = np.polyfit(np.log(4 * np.sin(lam / 2) ** 2),
                       np.log(np.maximum(periodogram[1:m + 1], 1e-12)), 1)[0]
    return float(np.clip(-slope, 0.0, .49))

def arma_asset(g, fractional=False):
    x = g.c_logrv_24.to_numpy(float)
    dates = g.timestamp.to_numpy()
    out = []
    fit = None
    raw_seen = []
    weights = None
    last_fit = -10_000
    for i in range(len(g) - 1):
        date = pd.Timestamp(dates[i])
        raw_seen.append(x[i])
        if date < pd.Timestamp("2023-01-01"):
            continue
        refit = fit is None or i - last_fit >= 60
        try:
            if refit:
                history = np.asarray(raw_seen)
                if fractional:
                    d = gph_d(history)
                    weights = fractional_weights(d, len(history))
                    z = np.convolve(history, weights, mode="full")[:len(history)]
                    z = z[len(weights) - 1:]
                    fit = ARIMA(z, order=(1, 0, 1), trend="c").fit()
                else:
                    fit = ARIMA(history, order=(1, 0, 1), trend="c").fit()
                last_fit = i
            elif fractional:
                recent = np.asarray(raw_seen[-len(weights):])[::-1]
                znew = float(np.dot(weights[:len(recent)], recent))
                fit = fit.append([znew], refit=False)
            else:
                fit = fit.append([x[i]], refit=False)
            fc = float(fit.forecast(1)[0])
            if fractional:
                recent = np.asarray(raw_seen[-(len(weights) - 1):])[::-1]
                fc -= float(np.dot(weights[1:len(recent) + 1], recent))
        except Exception:
            fc = float(np.mean(raw_seen[-60:]))
            fit = None
        out.append((date, g.symbol.iloc[0], x[i + 1], fc))
    return out

def garch_asset(g):
    r = 100 * g.ret_24.to_numpy(float)
    x = g.c_logrv_24.to_numpy(float)
    dates = g.timestamp.to_numpy()
    rows = []
    params = None
    variance = None
    last_fit = -10_000
    for i in range(len(g) - 1):
        date = pd.Timestamp(dates[i])
        if date < pd.Timestamp("2023-01-01"):
            continue
        try:
            if params is None or i - last_fit >= 60:
                res = arch_model(r[:i + 1], mean="Zero", vol="GARCH", p=1, q=1,
                                 dist="t", rescale=False).fit(disp="off", show_warning=False)
                params = res.params
                variance = float(res.forecast(horizon=1, reindex=False).variance.values[-1, 0])
                last_fit = i
            else:
                variance = (float(params["omega"]) + float(params["alpha[1]"]) * r[i] ** 2
                            + float(params["beta[1]"]) * variance)
            fc = float(np.log(max(variance / 10000, 1e-12)))
        except Exception:
            fc = float(np.mean(x[max(0, i - 60):i + 1]))
            params = None
        rows.append((date, g.symbol.iloc[0], x[i + 1], fc))
    return rows

def baseline_predictions():
    df = core.load_and_audit().sort_values(["symbol", "timestamp"])
    frames = {}
    for name, func in [("ARMA_1_1", lambda g: arma_asset(g, False)),
                       ("ARFIMA_1_d_1", lambda g: arma_asset(g, True)),
                       ("GARCH_1_1_t", garch_asset)]:
        rows = []
        for symbol, g in df.groupby("symbol"):
            print(f"{name}: {symbol}", flush=True)
            rows.extend(func(g.reset_index(drop=True)))
        frames[name] = pd.DataFrame(rows, columns=["timestamp", "symbol", "y_check", name])

    saved = pd.read_csv(PACKAGE / "final_results" / "holdout_predictions.csv",
                        parse_dates=["timestamp"])
    merged = None
    calibration_rows = []
    for name, frame in frames.items():
        val = frame[(frame.timestamp >= "2023-01-01") & (frame.timestamp < "2024-01-01")]
        X = np.column_stack([np.ones(len(val)), val[name].to_numpy(float)])
        coef = np.linalg.lstsq(X, val.y_check.to_numpy(float), rcond=None)[0]
        frame[name] = coef[0] + coef[1] * frame[name]
        calibration_rows.append({"model": name, "intercept": coef[0], "slope": coef[1],
                                 "calibration_start": "2023-01-01", "calibration_end": "2023-12-31"})
        hold = frame[frame.timestamp >= "2024-01-01"][["timestamp", "symbol", name]]
        merged = hold if merged is None else merged.merge(hold, on=["timestamp", "symbol"], how="inner")
    merged = saved.merge(merged, on=["timestamp", "symbol"], how="inner")
    rows = [score(merged, "HPTS_Final")]
    for name in frames:
        rows.append(score(merged, name))
        rows.append(score(merged, "HPTS_Final", name))
    pd.DataFrame(rows).to_csv(OUT / "hoang_baur_baselines.csv", index=False)
    pd.DataFrame(calibration_rows).to_csv(OUT / "baseline_validation_calibration.csv", index=False)
    merged[["timestamp", "symbol", "y", "HAR", "HPTS_Final", *frames]].to_csv(
        OUT / "baseline_holdout_predictions.csv", index=False)
    print(pd.DataFrame(rows).to_string(index=False), flush=True)

def horizon_data(days):
    df = core.load_and_audit().copy()
    if days == 1:
        df["target_end"] = df.timestamp + pd.Timedelta(days=1)
        return df
    target = {3: "y_3d", 7: "y_7d"}[days]
    if target in df:
        df = df.drop(columns="y").rename(columns={target: "y"})
    else:
        col = {3: "c_logrv_72", 7: "c_logrv_168"}[days]
        if not RAW_PANEL.exists():
            raise FileNotFoundError(
                f"{target} is absent from model_data.csv and the optional raw panel was not found at "
                f"{RAW_PANEL}. Set HPTS_RAW_PANEL to a compatible panel_hourly.parquet file."
            )
        raw = pd.read_parquet(RAW_PANEL, columns=["timestamp", "symbol", col])
        future = raw[raw.timestamp.dt.hour.eq(0)][["timestamp", "symbol", col]].copy()
        future["timestamp"] -= pd.Timedelta(days=days)
        future = future.rename(columns={col: "y_h"})
        df = df.drop(columns="y").merge(future, on=["timestamp", "symbol"], how="inner")
        df = df.rename(columns={"y_h": "y"})
    df["target_end"] = df.timestamp + pd.Timedelta(days=days)
    return df.dropna(subset=["y"]).sort_values(["timestamp", "symbol"]).reset_index(drop=True)

def rolling_panel_gap(df, features, deviations, hp, name, days):
    rows = []
    dates = np.array(sorted(df[df.timestamp >= "2024-01-01"].timestamp.unique()))
    for start in range(0, len(dates), core.REFIT_DAYS):
        block = dates[start:start + core.REFIT_DAYS]
        cutoff = pd.Timestamp(block[0])
        train = df[df.target_end <= cutoff].reset_index(drop=True)
        test = df[df.timestamp.isin(block)].reset_index(drop=True)
        pred = core.panel_predict(train, test, features, deviations, hp)
        rows.extend(zip(test.timestamp, test.symbol, test.y, pred))
    return pd.DataFrame(rows, columns=["timestamp", "symbol", "y", name])

def rolling_har_gap(df, days):
    rows = []
    dates = np.array(sorted(df[df.timestamp >= "2024-01-01"].timestamp.unique()))
    for start in range(0, len(dates), core.REFIT_DAYS):
        block = dates[start:start + core.REFIT_DAYS]
        cutoff = pd.Timestamp(block[0])
        train = df[df.target_end <= cutoff]
        test = df[df.timestamp.isin(block)]
        for symbol, tr in train.groupby("symbol"):
            te = test[test.symbol == symbol]
            if te.empty:
                continue
            x, z = core.standardise(tr, te, core.HAR)
            y = tr.y.to_numpy(float)
            beta = np.linalg.solve(x.T @ x + .1 * np.eye(x.shape[1]), x.T @ (y - y.mean()))
            pred = y.mean() + z @ beta
            rows.extend(zip(te.timestamp, te.symbol, te.y, pred))
    return pd.DataFrame(rows, columns=["timestamp", "symbol", "y", "HAR"])

def multihorizon():
    all_scores = []
    all_pred = []
    for days in (1, 3, 7):
        print(f"multi-horizon: {days} day(s)", flush=True)
        if days == 1:
            pred = pd.read_csv(PACKAGE / "final_results" / "holdout_predictions.csv",
                               parse_dates=["timestamp"])[
                ["timestamp", "symbol", "y", "HAR", "HPTS_NoIV", "HPTS_Final"]]
        else:
            df = horizon_data(days)
            har = rolling_har_gap(df, days)
            noiv = rolling_panel_gap(df, core.SPOT, core.SPOT,
                                     {"alpha": 10.0, "pool": 30.0}, "HPTS_NoIV", days)
            final = rolling_panel_gap(df, core.FULL, core.FULL,
                                     {"alpha": 100.0, "pool": 10.0}, "HPTS_Final", days)
            pred = har.merge(noiv.drop(columns="y"), on=["timestamp", "symbol"], how="inner")
            pred = pred.merge(final.drop(columns="y"), on=["timestamp", "symbol"], how="inner")
        pred["horizon_days"] = days
        all_pred.append(pred)
        for model, comparator in [("HPTS_NoIV", "HAR"), ("HPTS_Final", "HAR"),
                                  ("HPTS_Final", "HPTS_NoIV")]:
            row = score(pred, model, comparator, days)
            row["horizon_days"] = days
            all_scores.append(row)
    pd.DataFrame(all_scores).to_csv(OUT / "multihorizon_scores.csv", index=False)
    pd.concat(all_pred, ignore_index=True).to_csv(OUT / "multihorizon_predictions.csv", index=False)
    print(pd.DataFrame(all_scores).to_string(index=False), flush=True)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("task", choices=["permutation", "baselines", "multihorizon", "all"])
    args = parser.parse_args()
    OUT.mkdir(parents=True, exist_ok=True)
    if args.task in ("permutation", "all"):
        repeated_permutation()
    if args.task in ("baselines", "all"):
        baseline_predictions()
    if args.task in ("multihorizon", "all"):
        multihorizon()


In [ ]:
baseline_predictions()
multihorizon()
repeated_permutation()

In [ ]:
requested_dir = Path.cwd() / 'requested_tests'
display(pd.read_csv(requested_dir / 'hoang_baur_baselines.csv'))
display(pd.read_csv(requested_dir / 'multihorizon_scores.csv'))
display(pd.DataFrame([json.loads((requested_dir / 'permutation_summary.json').read_text(encoding='utf-8'))]))